# 📖 Konverter Quranic Arabic Corpus ke Database SQL

Selamat datang di Google Colab Notebook untuk pembuat database morfologi Quranic Arabic Corpus!

Notebook ini memproses penjelasan morfologi dan sintaksis kata-demi-kata dari **Quranic Arabic Corpus (versi 0.4)** dan mengonversinya menjadi **Database SQLite** relasional yang terstruktur dan terindeks (`quran_morphology.db`).

Dikembangkan untuk mempermudah penelitian linguistik, pencarian akar kata, analisis kata, dan pencarian teks lengkap secara langsung di Google Colab.

--- 
## 📤 Langkah 1: Unggah Dataset Morfologi

Jalankan sel di bawah ini untuk mengunggah berkas dataset `quranic-corpus-morphology-0.4.txt` dari komputer lokal Anda langsung ke lingkungan Google Colab.

In [ ]:
from google.colab import files
import os

dataset_filename = 'quranic-corpus-morphology-0.4.txt'

if not os.path.exists(dataset_filename):
    print(f"Silakan klik 'Choose Files' dan pilih berkas '{dataset_filename}':")
    uploaded = files.upload()
else:
    print(f"Dataset '{dataset_filename}' sudah tersedia di lingkungan kerja.")

--- 
## ⚙️ Langkah 2: Bangun Database SQLite

Sel ini akan menganalisis berkas morfologi baris-demi-baris, menerjemahkan kode Buckwalter ke Unicode Arab, menentukan atribut tata bahasa, mengunduh metadata Surah dan terjemahan bahasa Inggris Sahih International, serta memasukkannya ke dalam skema SQLite.

In [ ]:
import os
import re
import sqlite3
import urllib.request
import json

# Define paths
WORKSPACE_DIR = os.path.dirname(os.path.abspath(__file__))
CORPUS_FILE_PATH = os.path.join(WORKSPACE_DIR, "quranic-corpus-morphology-0.4.txt")
DB_FILE_PATH = os.path.join(WORKSPACE_DIR, "quran_morphology.db")

# Extended Buckwalter to Arabic Unicode Mapping
BUCKWALTER_TO_ARABIC = {
    # Letters
    'A': '\u0627',  # Alif
    'b': '\u0628',  # Ba
    't': '\u062a',  # Ta
    'v': '\u062b',  # Tha
    'j': '\u062c',  # Jeem
    'H': '\u062d',  # Hah
    'x': '\u062e',  # Khah
    'd': '\u062f',  # Dal
    '*': '\u0630',  # Thal
    'r': '\u0631',  # Ra
    'z': '\u0632',  # Zayn
    's': '\u0633',  # Seen
    '$': '\u0634',  # Sheen
    'S': '\u0635',  # Sad
    'D': '\u0636',  # Dad
    'T': '\u0637',  # Tah
    'Z': '\u0638',  # Zah
    'E': '\u0639',  # Ayn
    'g': '\u063a',  # Ghayn
    'f': '\u0641',  # Fa
    'q': '\u0642',  # Qaf
    'k': '\u0643',  # Kaf
    'l': '\u0644',  # Lam
    'm': '\u0645',  # Meem
    'n': '\u0646',  # Noon
    'h': '\u0647',  # Ha
    'w': '\u0648',  # Waw
    'y': '\u064a',  # Ya
    'Y': '\u0649',  # Alif Maksura
    'p': '\u0629',  # Ta Marbuta
    
    # Hamzas & variants
    '\'': '\u0621', # Hamza (standalone)
    '|': '\u0622',  # Alif Madda
    '>': '\u0623',  # Alif Hamza Above
    '&': '\u0624',  # Waw Hamza
    '<': '\u0625',  # Alif Hamza Below
    '}': '\u0626',  # Ya Hamza
    
    # Diacritics (Harakaat)
    'a': '\u064e',  # Fatha
    'u': '\u064f',  # Damma
    'i': '\u0650',  # Kasra
    'F': '\u064b',  # Fathatayn
    'N': '\u064c',  # Dammatayn
    'K': '\u064d',  # Kasratayn
    '~': '\u0651',  # Shadda
    'o': '\u0652',  # Sukun
    
    # Extended Quranic Symbols
    '`': '\u0670',  # Superscript Alif
    '{': '\u0671',  # Alif Wasla
    '^': '\u0653',  # Maddah Above
    '#': '\u0654',  # Hamza Above
    ':': '\u06e2',  # Small High Seen
    '@': '\u06df',  # Small High Rounded Zero
    '"': '\u06e0',  # Small High Upright Rectangular Zero
    '[': '\u06e8',  # Small High Meem Isolated
    ';': '\u06e5',  # Small Low Seen
    ',': '\u06e5',  # Small Waw
    '.': '\u06e6',  # Small Ya
    '!': '\u06e7',  # Small High Noon
    '-': '\u06ea',  # Empty Centre Low Stop
    '+': '\u06e9',  # Empty Centre High Stop
    '%': '\u06eb',  # Rounded High Stop (Filled Centre)
    ']': '\u06ed',  # Small Low Meem
    '_': '\u0640',  # Tatweel
}

SURAH_FALLBACK_NAMES = [
    "Al-Fatihah", "Al-Baqarah", "Ali 'Imran", "An-Nisa", "Al-Ma'idah", "Al-An'am", "Al-A'raf", "Al-Anfal", "At-Tawbah", "Yunus",
    "Hud", "Yusuf", "Ar-Ra'd", "Ibrahim", "Al-Hijr", "An-Nahl", "Al-Isra", "Al-Kahf", "Maryam", "Ta-Ha",
    "Al-Anbiya", "Al-Hajj", "Al-Mu'minun", "An-Nur", "Al-Furqan", "Ash-Shu'ara", "An-Naml", "Al-Qasas", "Al-Ankabut", "Ar-Rum",
    "Luqman", "As-Sajdah", "Al-Ahzab", "Saba", "Fatir", "Ya-Sin", "As-Saffat", "Sad", "Az-Zumar", "Ghafir",
    "Fussilat", "Ash-Shura", "Az-Zukhruf", "Ad-Dukhan", "Al-Jathiyah", "Al-Ahqaf", "Muhammad", "Al-Fath", "Al-Hujurat", "Qaf",
    "Adh-Dhariyat", "At-Tur", "An-Najm", "Al-Qamar", "Ar-Rahman", "Al-Waqi'ah", "Al-Hadid", "Al-Mujadilah", "Al-Hashr", "Al-Mumtahanah",
    "As-Saff", "Al-Jumu'ah", "Al-Munafiqun", "At-Taghabun", "At-Talaq", "At-Tahrim", "Al-Mulk", "Al-Qalam", "Al-Haqqah", "Al-Ma'arij",
    "Nuh", "Al-Jinn", "Al-Muzzammil", "Al-Muddaththir", "Al-Qiyamah", "Al-Insan", "Al-Mursalat", "An-Naba", "An-Nazi'at", "Abasa",
    "At-Takwir", "Al-Infitar", "Al-Mutaffifin", "Al-Inshiqaq", "Al-Buruj", "At-Tariq", "Al-A'la", "Al-Ghashiyah", "Al-Fajr", "Al-Balad",
    "Ash-Shams", "Al-Layl", "Ad-Duha", "Ash-Sharh", "At-Tin", "Al-Alaq", "Al-Qadr", "Al-Bayyinah", "Az-Zalzalah", "Al-Adiyat",
    "Al-Qari'ah", "At-Takathur", "Al-Asr", "Al-Humazah", "Al-Fil", "Quraysh", "Al-Ma'un", "Al-Kawthar", "Al-Kafirun", "An-Nasr",
    "Al-Masad", "Al-Ikhlas", "Al-Falaq", "An-Nas"
]

def buckwalter_to_arabic(text):
    if not text:
        return ""
    return "".join(BUCKWALTER_TO_ARABIC.get(char, char) for char in text)

def normalize_arabic(text):
    if not text:
        return ""
    # Strip all Arabic diacritics / harakaat (range \u064b to \u0652, superscript alif \u0670, hamza/madda marks \u0653-\u0655)
    diacritics_pattern = re.compile(r'[\u064b-\u0652\u0670\u0653\u0654\u0655]')
    text = diacritics_pattern.sub('', text)
    
    # Normalize Alifs: ٱ (alif wasla), آ (alif madda), أ (alif hamza above), إ (alif hamza below) to standard Alif ا
    text = re.sub(r'[ٱآأإ]', '\u0627', text)
    
    # Normalize Ya Hamza ئ and Alif Maksura ى to standard Ya ي
    text = re.sub(r'[ئى]', '\u064a', text)
    
    # Normalize Ta Marbuta ة to Ha ه
    text = re.sub(r'ة', '\u0647', text)
    
    return text

def fetch_surah_metadata():
    print("Fetching Surah metadata from api.alquran.cloud...")
    try:
        url = "http://api.alquran.cloud/v1/surah"
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=10) as response:
            res = json.loads(response.read().decode('utf-8'))
            if res.get('code') == 200:
                print("Successfully fetched Surah metadata.")
                return res['data']
    except Exception as e:
        print(f"Failed to fetch Surah metadata ({e}). Using offline fallbacks.")
    return None

def fetch_translation():
    print("Fetching Sahih International English translation from Tanzil.info...")
    translation_dict = {}
    try:
        url = "https://tanzil.info/trans/en.sahih"
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=15) as response:
            lines = response.read().decode('utf-8').splitlines()
            for line in lines:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                parts = line.split('|')
                if len(parts) >= 3:
                    s_num, v_num, trans_text = int(parts[0]), int(parts[1]), parts[2]
                    translation_dict[(s_num, v_num)] = trans_text
            print(f"Successfully fetched {len(translation_dict)} verse translations.")
    except Exception as e:
        print(f"Failed to fetch translation ({e}). Proceeding without translation.")
    return translation_dict

def parse_pronoun_features(pron_val, info):
    if not pron_val:
        return
    info['person'] = pron_val[0]
    if len(pron_val) == 3:
        info['gender'] = pron_val[1]
        info['number'] = pron_val[2]
    elif len(pron_val) == 2:
        if pron_val[1] in ('M', 'F'):
            info['gender'] = pron_val[1]
        elif pron_val[1] in ('S', 'D', 'P'):
            info['number'] = pron_val[1]

def parse_features(features_str):
    parts = features_str.split('|')
    seg_type = parts[0]
    info = {
        'segment_type': seg_type,
        'pos': None,
        'lemma': None,
        'root': None,
        'gender': None,
        'number': None,
        'person': None,
        'case_state': None,
        'aspect_mood': None,
        'voice': None,
        'form_derived': None
    }
    
    for part in parts[1:]:
        if ':' in part:
            key, val = part.split(':', 1)
            if key == 'POS':
                info['pos'] = val
            elif key == 'LEM':
                info['lemma'] = val
            elif key == 'ROOT':
                info['root'] = val
            elif key == 'PRON':
                parse_pronoun_features(val, info)
            elif key == 'SP':
                pass
        else:
            if part in ('NOM', 'ACC', 'GEN'):
                info['case_state'] = part
            elif part == 'INDEF':
                info['case_state'] = (info['case_state'] + '+INDEF') if info['case_state'] else 'INDEF'
            elif part in ('M', 'F'):
                info['gender'] = part
            elif part in ('S', 'D', 'P'):
                info['number'] = part
            elif part in ('1', '2', '3'):
                info['person'] = part
            elif part in ('PERF', 'IMPF', 'IMPV'):
                info['aspect_mood'] = part
            elif part in ('JUS', 'SUBJ'):
                info['aspect_mood'] = (info['aspect_mood'] + f'+{part}') if info['aspect_mood'] else part
            elif part in ('ACT', 'PASS'):
                info['voice'] = part
            elif part.startswith('(') and part.endswith(')'):
                info['form_derived'] = part[1:-1]
            elif len(part) in (2, 3) and part[0] in ('1', '2', '3'):
                info['person'] = part[0]
                if len(part) == 3:
                    info['gender'] = part[1]
                    info['number'] = part[2]
                elif len(part) == 2:
                    if part[1] in ('M', 'F'):
                        info['gender'] = part[1]
                    elif part[1] in ('S', 'D', 'P'):
                        info['number'] = part[1]
    return info

def create_database(conn):
    cursor = conn.cursor()
    
    # Drop existing tables to recreate with new schema
    cursor.execute("DROP TABLE IF EXISTS tokens;")
    cursor.execute("DROP TABLE IF EXISTS words;")
    cursor.execute("DROP TABLE IF EXISTS verses;")
    cursor.execute("DROP TABLE IF EXISTS suras;")
    cursor.execute("DROP TABLE IF EXISTS roots;")
    cursor.execute("DROP TABLE IF EXISTS verses_fts;")
    
    # 1. Suras Table
    cursor.execute("""
    CREATE TABLE suras (
        id INTEGER PRIMARY KEY,
        name_arabic TEXT NOT NULL,
        name_english TEXT NOT NULL,
        translation TEXT NOT NULL,
        type TEXT NOT NULL,
        total_verses INTEGER NOT NULL
    );
    """)
    
    # 2. Verses Table
    cursor.execute("""
    CREATE TABLE verses (
        id INTEGER PRIMARY KEY,
        sura_id INTEGER NOT NULL,
        verse_num INTEGER NOT NULL,
        text_arabic TEXT NOT NULL,
        text_arabic_normalized TEXT NOT NULL,
        text_transliterated TEXT NOT NULL,
        translation TEXT,
        FOREIGN KEY (sura_id) REFERENCES suras(id),
        UNIQUE(sura_id, verse_num)
    );
    """)
    
    # 3. Words Table
    cursor.execute("""
    CREATE TABLE words (
        id INTEGER PRIMARY KEY,
        sura_id INTEGER NOT NULL,
        verse_num INTEGER NOT NULL,
        word_num INTEGER NOT NULL,
        text_arabic TEXT NOT NULL,
        text_arabic_normalized TEXT NOT NULL,
        text_transliterated TEXT NOT NULL,
        part_of_speech_brief TEXT NOT NULL,
        FOREIGN KEY (sura_id, verse_num) REFERENCES verses(sura_id, verse_num)
    );
    """)
    
    # 4. Tokens Table
    cursor.execute("""
    CREATE TABLE tokens (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        sura_id INTEGER NOT NULL,
        verse_num INTEGER NOT NULL,
        word_num INTEGER NOT NULL,
        token_num INTEGER NOT NULL,
        segment_type TEXT NOT NULL,
        form_transliterated TEXT NOT NULL,
        form_arabic TEXT NOT NULL,
        tag TEXT NOT NULL,
        features TEXT NOT NULL,
        pos TEXT,
        lemma_transliterated TEXT,
        lemma_arabic TEXT,
        root_transliterated TEXT,
        root_arabic TEXT,
        gender TEXT,
        number TEXT,
        person TEXT,
        case_state TEXT,
        aspect_mood TEXT,
        voice TEXT,
        form_derived TEXT,
        FOREIGN KEY (sura_id, verse_num, word_num) REFERENCES words(sura_id, verse_num, word_num),
        FOREIGN KEY (root_transliterated) REFERENCES roots(root_transliterated)
    );
    """)
    
    # 5. Roots Table
    cursor.execute("""
    CREATE TABLE roots (
        root_transliterated TEXT PRIMARY KEY,
        root_arabic TEXT NOT NULL,
        occurrence_count INTEGER DEFAULT 0
    );
    """)
    
    # 6. FTS5 Virtual Table
    cursor.execute("""
    CREATE VIRTUAL TABLE verses_fts USING fts5(
        sura_id UNINDEXED,
        verse_num UNINDEXED,
        text_arabic,
        text_arabic_normalized,
        text_transliterated,
        translation
    );
    """)
    
    # Indexes for optimization
    cursor.execute("CREATE INDEX idx_verses_sura_verse ON verses(sura_id, verse_num);")
    cursor.execute("CREATE INDEX idx_words_sura_verse_word ON words(sura_id, verse_num, word_num);")
    cursor.execute("CREATE INDEX idx_tokens_sura_verse_word ON tokens(sura_id, verse_num, word_num);")
    cursor.execute("CREATE INDEX idx_tokens_root ON tokens(root_transliterated);")
    
    conn.commit()

def main():
    if not os.path.exists(CORPUS_FILE_PATH):
        print(f"Error: Corpus file not found at {CORPUS_FILE_PATH}")
        return

    # Fetch external metadata & translation
    suras_meta = fetch_surah_metadata()
    translation_dict = fetch_translation()
    
    print("Connecting to database...")
    conn = sqlite3.connect(DB_FILE_PATH)
    create_database(conn)
    cursor = conn.cursor()
    
    # Parse corpus file
    print("Parsing Quranic corpus morphology file...")
    
    word_map = {}
    roots_set = set()
    roots_count = {}
    surah_max_verses = {}
    
    line_count = 0
    with open(CORPUS_FILE_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line_count += 1
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            
            # Read columns
            parts = line.split("\t")
            if len(parts) < 4:
                continue
                
            location_str = parts[0].strip()
            form_trans = parts[1].strip()
            tag = parts[2].strip()
            features_str = parts[3].strip()
            
            # Parse Location e.g. (1:1:1:1)
            loc_match = re.match(r"\((\d+):(\d+):(\d+):(\d+)\)", location_str)
            if not loc_match:
                continue
                
            s_num, v_num, w_num, t_num = map(int, loc_match.groups())
            
            # Track max verses for Surah metadata
            surah_max_verses[s_num] = max(surah_max_verses.get(s_num, 0), v_num)
            
            # Parse features
            feat_info = parse_features(features_str)
            
            # Translate form, lemma, root
            form_arabic = buckwalter_to_arabic(form_trans)
            lemma_arabic = buckwalter_to_arabic(feat_info['lemma']) if feat_info['lemma'] else None
            
            root_trans = feat_info['root']
            root_arabic = None
            if root_trans:
                root_arabic = buckwalter_to_arabic(root_trans)
                roots_set.add((root_trans, root_arabic))
                roots_count[root_trans] = roots_count.get(root_trans, 0) + 1
            
            token_rec = {
                'sura_id': s_num,
                'verse_num': v_num,
                'word_num': w_num,
                'token_num': t_num,
                'segment_type': feat_info['segment_type'],
                'form_transliterated': form_trans,
                'form_arabic': form_arabic,
                'tag': tag,
                'features': features_str,
                'pos': feat_info['pos'],
                'lemma_transliterated': feat_info['lemma'],
                'lemma_arabic': lemma_arabic,
                'root_transliterated': root_trans,
                'root_arabic': root_arabic,
                'gender': feat_info['gender'],
                'number': feat_info['number'],
                'person': feat_info['person'],
                'case_state': feat_info['case_state'],
                'aspect_mood': feat_info['aspect_mood'],
                'voice': feat_info['voice'],
                'form_derived': feat_info['form_derived']
            }
            
            key = (s_num, v_num, w_num)
            if key not in word_map:
                word_map[key] = []
            word_map[key].append(token_rec)
            
            if line_count % 30000 == 0:
                print(f"Processed {line_count} lines...")
                
    print(f"Finished parsing. Total unique words parsed: {len(word_map)}")
    
    # 1. Populate roots
    print("Inserting roots...")
    root_records = []
    for r_trans, r_ar in roots_set:
        count = roots_count.get(r_trans, 0)
        root_records.append((r_trans, r_ar, count))
    cursor.executemany("INSERT INTO roots (root_transliterated, root_arabic, occurrence_count) VALUES (?, ?, ?);", root_records)
    conn.commit()
    
    # 2. Populate suras
    print("Inserting Surah metadata...")
    sura_records = []
    for s_id in range(1, 115):
        tot_v = surah_max_verses.get(s_id, 0)
        name_ar = f"سورة {s_id}"
        name_en = SURAH_FALLBACK_NAMES[s_id - 1]
        trans_en = "Translation"
        rev_type = "Meccan"
        
        if suras_meta and s_id - 1 < len(suras_meta):
            meta = suras_meta[s_id - 1]
            name_ar = meta.get('name', name_ar)
            name_en = meta.get('transliteration_en', name_en)
            trans_en = meta.get('translation_en', trans_en)
            rev_type = meta.get('revelationType', rev_type)
            if 'numberOfAyahs' in meta:
                tot_v = meta['numberOfAyahs']
                
        sura_records.append((s_id, name_ar, name_en, trans_en, rev_type, tot_v))
    cursor.executemany("INSERT INTO suras (id, name_arabic, name_english, translation, type, total_verses) VALUES (?, ?, ?, ?, ?, ?);", sura_records)
    conn.commit()
    
    # 3. Aggregate words, verses and tokens
    print("Reconstructing words and verses...")
    words_records = []
    tokens_records = []
    verse_map = {}
    
    sorted_word_keys = sorted(word_map.keys())
    
    for s_num, v_num, w_num in sorted_word_keys:
        tokens_in_word = sorted(word_map[(s_num, v_num, w_num)], key=lambda x: x['token_num'])
        
        word_trans = "".join(t['form_transliterated'] for t in tokens_in_word)
        word_ar = "".join(t['form_arabic'] for t in tokens_in_word)
        word_ar_norm = normalize_arabic(word_ar)
        
        pos_brief = "+".join(t['tag'] for t in tokens_in_word)
        word_id = s_num * 1000000 + v_num * 1000 + w_num
        words_records.append((word_id, s_num, v_num, w_num, word_ar, word_ar_norm, word_trans, pos_brief))
        
        for t in tokens_in_word:
            tokens_records.append((
                s_num, v_num, w_num, t['token_num'], t['segment_type'],
                t['form_transliterated'], t['form_arabic'], t['tag'], t['features'],
                t['pos'], t['lemma_transliterated'], t['lemma_arabic'],
                t['root_transliterated'], t['root_arabic'],
                t['gender'], t['number'], t['person'], t['case_state'],
                t['aspect_mood'], t['voice'], t['form_derived']
            ))
            
        v_key = (s_num, v_num)
        if v_key not in verse_map:
            verse_map[v_key] = {'words_trans': [], 'words_ar': []}
        verse_map[v_key]['words_trans'].append(word_trans)
        verse_map[v_key]['words_ar'].append(word_ar)
        
    print(f"Inserting {len(words_records)} words into table...")
    cursor.executemany("""
        INSERT INTO words (id, sura_id, verse_num, word_num, text_arabic, text_arabic_normalized, text_transliterated, part_of_speech_brief)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?);
    """, words_records)
    conn.commit()
    
    print(f"Inserting {len(tokens_records)} tokens into table...")
    cursor.executemany("""
        INSERT INTO tokens (
            sura_id, verse_num, word_num, token_num, segment_type,
            form_transliterated, form_arabic, tag, features,
            pos, lemma_transliterated, lemma_arabic,
            root_transliterated, root_arabic,
            gender, number, person, case_state,
            aspect_mood, voice, form_derived
        ) VALUES (
            ?, ?, ?, ?, ?,
            ?, ?, ?, ?,
            ?, ?, ?,
            ?, ?,
            ?, ?, ?, ?,
            ?, ?, ?
        );
    """, tokens_records)
    conn.commit()
    
    # 4. Insert Verses and populate FTS5
    print("Reconstructing and inserting verses...")
    verses_records = []
    sorted_verse_keys = sorted(verse_map.keys())
    
    for s_num, v_num in sorted_verse_keys:
        v_id = s_num * 1000 + v_num
        text_ar = " ".join(verse_map[(s_num, v_num)]['words_ar'])
        text_ar_norm = normalize_arabic(text_ar)
        text_trans = " ".join(verse_map[(s_num, v_num)]['words_trans'])
        translation = translation_dict.get((s_num, v_num), None)
        
        verses_records.append((v_id, s_num, v_num, text_ar, text_ar_norm, text_trans, translation))
        
    cursor.executemany("""
        INSERT INTO verses (id, sura_id, verse_num, text_arabic, text_arabic_normalized, text_transliterated, translation)
        VALUES (?, ?, ?, ?, ?, ?, ?);
    """, verses_records)
    conn.commit()
    
    # 5. Populate FTS5 table
    print("Populating Full-Text Search (FTS5) table...")
    cursor.execute("""
    INSERT INTO verses_fts (sura_id, verse_num, text_arabic, text_arabic_normalized, text_transliterated, translation)
    SELECT sura_id, verse_num, text_arabic, text_arabic_normalized, text_transliterated, IFNULL(translation, '') FROM verses;
    """)
    conn.commit()
    
    # Done! Summarize stats
    cursor.execute("SELECT COUNT(*) FROM suras;")
    num_suras = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(*) FROM verses;")
    num_verses = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(*) FROM words;")
    num_words = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(*) FROM tokens;")
    num_tokens = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(*) FROM roots;")
    num_roots = cursor.fetchone()[0]
    
    print("\n" + "="*40)
    print("DATABASE BUILD COMPLETE SUCCESSFUL!")
    print(f"Database saved to: {DB_FILE_PATH}")
    print(f"Total Suras loaded:      {num_suras}")
    print(f"Total Verses loaded:     {num_verses}")
    print(f"Total Words loaded:      {num_words}")
    print(f"Total Tokens loaded:     {num_tokens}")
    print(f"Total Unique Roots:      {num_roots}")
    print("="*40 + "\n")
    
    conn.close()

if __name__ == "__main__":
    main()


--- 
## 🧰 Langkah 3: Definisikan Fungsi Pembantu Pencarian & Kueri

Mari muat fungsi pembantu kueri SQL. Ini mencakup fungsi `normalize_arabic` untuk pencarian tanpa harakat, kueri akar kata, pencarian frasa, dan peninjauan morfologi ayat.

In [ ]:
import os
import sqlite3
import sys
import re

# Define path to the database
WORKSPACE_DIR = os.path.dirname(os.path.abspath(__file__))
DB_FILE_PATH = os.path.join(WORKSPACE_DIR, "quran_morphology.db")

def print_section(title):
    print("\n" + "="*80)
    print(f" {title} ".center(80, "="))
    print("="*80)

def normalize_arabic(text):
    if not text:
        return ""
    # Strip all Arabic diacritics / harakaat
    diacritics_pattern = re.compile(r'[\u064b-\u0652\u0670\u0653\u0654\u0655]')
    text = diacritics_pattern.sub('', text)
    
    # Normalize Alifs
    text = re.sub(r'[ٱآأإ]', '\u0627', text)
    
    # Normalize Ya Hamza and Alif Maksura to standard Ya
    text = re.sub(r'[ئى]', '\u064a', text)
    
    # Normalize Ta Marbuta to Ha
    text = re.sub(r'ة', '\u0647', text)
    
    return text

def query_by_root(conn, root_ar):
    print_section(f"Querying Words with Root: {root_ar}")
    cursor = conn.cursor()
    cursor.execute("""
        SELECT 
            t.sura_id, t.verse_num, t.word_num,
            s.name_english,
            w.text_arabic AS word_arabic,
            t.form_arabic AS segment_arabic,
            t.tag,
            t.features,
            v.translation
        FROM tokens t
        JOIN words w ON t.sura_id = w.sura_id AND t.verse_num = w.verse_num AND t.word_num = w.word_num
        JOIN verses v ON t.sura_id = v.sura_id AND t.verse_num = v.verse_num
        JOIN suras s ON t.sura_id = s.id
        WHERE t.root_arabic = ? OR t.root_transliterated = ?
        LIMIT 10;
    """, (root_ar, root_ar))
    
    results = cursor.fetchall()
    if not results:
        print("No matches found for root.")
        return
        
    print(f"{'Sura:Verse:Word':<18} | {'Surah Name':<12} | {'Word (AR)':<10} | {'Segment':<8} | {'Tag':<5} | {'Features'}")
    print("-"*80)
    for row in results:
        loc = f"{row[0]}:{row[1]}:{row[2]}"
        print(f"{loc:<18} | {row[3]:<12} | {row[4]:<10} | {row[5]:<8} | {row[6]:<5} | {row[7]}")
        print(f"   [Translation] {row[8]}\n")

def query_by_phrase(conn, phrase_ar):
    print_section(f"Searching for Arabic Phrase: {phrase_ar}")
    cursor = conn.cursor()
    
    # Normalize the query phrase first
    normalized_phrase = normalize_arabic(phrase_ar)
    print(f"(Normalized phrase query: '{normalized_phrase}')")
    
    # Search on text_arabic_normalized column in FTS5
    query_str = f'text_arabic_normalized:"{normalized_phrase}"'
    cursor.execute("""
        SELECT 
            v.sura_id, v.verse_num, 
            s.name_english,
            v.text_arabic, 
            v.translation
        FROM verses_fts f
        JOIN verses v ON f.sura_id = v.sura_id AND f.verse_num = v.verse_num
        JOIN suras s ON v.sura_id = s.id
        WHERE verses_fts MATCH ?
        LIMIT 5;
    """, (query_str,))
    
    results = cursor.fetchall()
    if not results:
        print("No matches found for phrase.")
        return
        
    for row in results:
        print(f"Surah {row[0]}:{row[1]} ({row[2]}):")
        print(f"   [Arabic]      {row[3]}")
        print(f"   [Translation] {row[4]}\n")

def search_english_translation(conn, keyword):
    print_section(f"FTS5 Search English Translation: '{keyword}'")
    cursor = conn.cursor()
    cursor.execute("""
        SELECT 
            v.sura_id, v.verse_num, 
            s.name_english,
            v.text_arabic, 
            v.translation
        FROM verses_fts f
        JOIN verses v ON f.sura_id = v.sura_id AND f.verse_num = v.verse_num
        JOIN suras s ON v.sura_id = s.id
        WHERE verses_fts MATCH ?
        LIMIT 5;
    """, (f"translation:{keyword}",))
    
    results = cursor.fetchall()
    if not results:
        print("No matches found.")
        return
        
    for row in results:
        print(f"Surah {row[0]}:{row[1]} ({row[2]}):")
        print(f"   [Arabic]      {row[3]}")
        print(f"   [Translation] {row[4]}\n")

def verse_morphology_breakdown(conn, sura, verse):
    print_section(f"Morphological Breakdown for Verse {sura}:{verse}")
    cursor = conn.cursor()
    
    cursor.execute("SELECT text_arabic, translation FROM verses WHERE sura_id = ? AND verse_num = ?;", (sura, verse))
    verse_info = cursor.fetchone()
    if not verse_info:
        print("Verse not found.")
        return
        
    print(f"Verse Arabic:      {verse_info[0]}")
    print(f"Verse Translation: {verse_info[1]}")
    print("-"*80)
    
    cursor.execute("""
        SELECT 
            word_num, token_num, segment_type,
            form_arabic, tag, pos, lemma_arabic, root_arabic, features
        FROM tokens
        WHERE sura_id = ? AND verse_num = ?
        ORDER BY word_num, token_num;
    """, (sura, verse))
    
    tokens = cursor.fetchall()
    current_word = None
    
    for t in tokens:
        word_num = t[0]
        if current_word != word_num:
            current_word = word_num
            print(f"\nWord {word_num}:")
            
        seg_num = t[1]
        seg_type = t[2]
        form_ar = t[3]
        tag = t[4]
        pos = t[5] if t[5] else '-'
        lemma = t[6] if t[6] else '-'
        root = t[7] if t[7] else '-'
        feat = t[8]
        
        print(f"  Token {seg_num} ({seg_type:<6}): {form_ar:<8} | Tag: {tag:<4} | POS: {pos:<4} | Lemma: {lemma:<8} | Root: {root:<6} | Features: {feat}")

def show_statistics(conn):
    print_section("Quranic Linguistics Statistics")
    cursor = conn.cursor()
    
    # 1. Top 10 Roots
    print("Top 10 Most Frequent Roots in the Quran:")
    cursor.execute("""
        SELECT root_arabic, root_transliterated, occurrence_count 
        FROM roots 
        WHERE root_arabic IS NOT NULL
        ORDER BY occurrence_count DESC 
        LIMIT 10;
    """)
    top_roots = cursor.fetchall()
    print(f"   {'Rank':<4} | {'Root (AR)':<10} | {'Root (BW)':<10} | {'Occurrences'}")
    print("   " + "-"*45)
    for i, row in enumerate(top_roots, 1):
        print(f"   {i:<4} | {row[0]:<10} | {row[1]:<10} | {row[2]}")
    print()
    
    # 2. POS Distribution
    print("Distribution of Major Part of Speech (POS) Tags:")
    cursor.execute("""
        SELECT tag, COUNT(*) as cnt 
        FROM tokens 
        GROUP BY tag 
        ORDER BY cnt DESC 
        LIMIT 10;
    """)
    pos_dist = cursor.fetchall()
    print(f"   {'Tag':<6} | {'Occurrences'}")
    print("   " + "-"*22)
    for row in pos_dist:
        print(f"   {row[0]:<6} | {row[1]}")
    print()

--- 
## 📊 Langkah 4: Tampilkan Statistik Umum Database

Jalankan sel di bawah ini untuk melihat statistik umum database, termasuk 10 akar kata paling sering muncul dan distribusi kelas kata.

In [ ]:
conn = sqlite3.connect(DB_FILE_PATH)
show_statistics(conn)
conn.close()

--- 
## 🔍 Langkah 5: Pencarian Interaktif Berdasarkan Akar Kata

Cari kata-kata di dalam Al-Quran yang berasal dari akar kata Arab tertentu.

*Fitur Colab: Gunakan kotak input formulir di sebelah kanan untuk memasukkan akar kata (misalnya, `رحm` atau `كتب` atau `علم`).*

In [ ]:
#@title Pencarian Akar Kata
root_to_search = "\u0631\u062d\u0645" #@param {type:"string"}

conn = sqlite3.connect(DB_FILE_PATH)
query_by_root(conn, root_to_search)
conn.close()

--- 
## 🗣️ Langkah 6: Pencarian Interaktif Frasa Arab

Cari urutan kata yang tepat menggunakan Pencarian Teks Lengkap (FTS5). Kueri secara otomatis dinormalisasi agar cocok dengan pencarian tulisan Arab modern biasa tanpa harakat.

*Fitur Colab: Masukkan frasa di kotak input teks (misalnya, `رب العلمin` atau `الحمد لله`).*

In [ ]:
#@title Pencarian Frasa Arab
phrase_to_search = "\u0631\u0628 \u0627\u0644\u0639\u0644\u0645\u064a\u0646" #@param {type:"string"}

conn = sqlite3.connect(DB_FILE_PATH)
query_by_phrase(conn, phrase_to_search)
conn.close()

--- 
## 📑 Langkah 7: Peninjau Interaktif Morfologi Ayat

Pilih Surah dan Ayat untuk melihat rincian morfologi kata-demi-kata dan token-demi-token secara lengkap.

*Fitur Colab: Geser slider atau masukkan angka di bawah ini untuk memilih ayat.*

In [ ]:
#@title Peninjau Morfologi Ayat
surah = 1 #@param {type:"slider", min:1, max:114, step:1}
verse = 1 #@param {type:"integer"}

conn = sqlite3.connect(DB_FILE_PATH)
verse_morphology_breakdown(conn, surah, verse)
conn.close()

--- 
## 🛠️ Langkah 8: Bangun Database Pembelajaran Kosakata (Learning Harness)

Silakan jalankan sel di bawah ini untuk membangun database pembelajaran kosakata `learning_harness.db`.

Sel ini akan mengunduh `quran_arabic_roots_lane_lexicon_2026-02-12.json` (jika tidak ada di lokal), memetakan arti dari Lane's Lexicon ke seluruh 1.642 akar kata Arab Al-Quran, menerjemahkannya ke bahasa Indonesia, dan menyimpannya dalam tabel SQL.

In [ ]:
import os
import sqlite3
import urllib.request
import urllib.parse
import json
import time
import re

# Define paths
WORKSPACE_DIR = os.path.dirname(os.path.abspath(__file__))
MORPHOLOGY_DB_PATH = os.path.join(WORKSPACE_DIR, "quran_morphology.db")
HARNESS_DB_PATH = os.path.join(WORKSPACE_DIR, "learning_harness.db")

# URL of the Lane's Lexicon etymology database JSON
LANE_JSON_URL = "https://raw.githubusercontent.com/aliozdenisik/quran-arabic-roots-lane-lexicon/main/quran_arabic_roots_lane_lexicon_2026-02-12.json"

def translate_en_to_id(text):
    if not text:
        return ""
    # Clean text to keep it short for vocabulary mapping
    # (Extract the first sentence or first few words if it's too long)
    match = re.match(r'^([^.;]+)', text)
    clean_text = match.group(1).strip() if match else text
    
    try:
        url = "https://translate.googleapis.com/translate_a/single?client=gtx&sl=en&tl=id&dt=t&q=" + urllib.parse.quote(clean_text)
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=5) as response:
            res = json.loads(response.read().decode('utf-8'))
            translated_text = "".join(segment[0] for segment in res[0] if segment[0])
            return translated_text.strip()
    except Exception as e:
        # Fallback to English if translation API fails
        return clean_text

LANE_JSON_FILE_PATH = os.path.join(WORKSPACE_DIR, "quran_arabic_roots_lane_lexicon_2026-02-12.json")

def download_lane_lexicon():
    # Try local load first
    if os.path.exists(LANE_JSON_FILE_PATH):
        print(f"Loading Lane's Lexicon from local file: {LANE_JSON_FILE_PATH}...")
        try:
            with open(LANE_JSON_FILE_PATH, "r", encoding="utf-8") as f:
                data = json.load(f)
            roots_list = data.get('roots', [])
            roots_dict = {}
            for entry in roots_list:
                rb = entry.get('root_buckwalter')
                if rb:
                    roots_dict[rb] = entry
            print("Successfully loaded Lane's Lexicon from local file.")
            return roots_dict
        except Exception as e:
            print(f"Failed to read local Lane's Lexicon file: {e}. Falling back to download...")
            
    print(f"Downloading Lane's Lexicon JSON database (approx. 11MB) from {LANE_JSON_URL}...")
    try:
        req = urllib.request.Request(LANE_JSON_URL, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=30) as response:
            data = json.loads(response.read().decode('utf-8'))
            print("Successfully downloaded Lane's Lexicon JSON.")
            
            # Save locally for future use
            try:
                with open(LANE_JSON_FILE_PATH, "w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False)
                print(f"Saved Lane's Lexicon locally to {LANE_JSON_FILE_PATH}")
            except Exception as e:
                print(f"Warning: Failed to save Lane's Lexicon locally: {e}")
                
            roots_list = data.get('roots', [])
            roots_dict = {}
            for entry in roots_list:
                rb = entry.get('root_buckwalter')
                if rb:
                    roots_dict[rb] = entry
            return roots_dict
    except Exception as e:
        print(f"Error downloading Lane's Lexicon database: {e}")
        return {}

def main():
    if not os.path.exists(MORPHOLOGY_DB_PATH):
        print(f"Error: Morphology database not found at {MORPHOLOGY_DB_PATH}")
        print("Please build it first using convert_corpus.py.")
        return
        
    # Download root vocabulary definitions
    lane_roots = download_lane_lexicon()
    if not lane_roots:
        print("Failed to load root definitions. Exiting.")
        return
        
    print("Connecting to morphology database to retrieve roots...")
    conn_morph = sqlite3.connect(MORPHOLOGY_DB_PATH)
    cursor_morph = conn_morph.cursor()
    
    # Get all roots and their transliterations from our build database
    cursor_morph.execute("SELECT root_arabic, root_transliterated, occurrence_count FROM roots WHERE root_arabic IS NOT NULL;")
    roots_in_db = cursor_morph.fetchall()
    conn_morph.close()
    
    total_roots = len(roots_in_db)
    print(f"Found {total_roots} roots in morphology database.")
    
    # Connect and initialize Learning Harness database
    print(f"Initializing learning harness database at: {HARNESS_DB_PATH}")
    conn_harness = sqlite3.connect(HARNESS_DB_PATH)
    cursor_harness = conn_harness.cursor()
    
    cursor_harness.execute("DROP TABLE IF EXISTS learning_harness;")
    cursor_harness.execute("""
        CREATE TABLE learning_harness (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            root TEXT NOT NULL,
            en_word TEXT,
            id_word TEXT
        );
    """)
    conn_harness.commit()
    
    # Process and insert root vocabulary
    print("Processing roots and translating definitions to Indonesian...")
    insert_records = []
    
    start_time = time.time()
    for idx, (root_ar, root_trans, count) in enumerate(roots_in_db, 1):
        # Look up root details in the downloaded Lane's Lexicon
        # The key in lane_roots dictionary matches the transliterated root (or root_trans)
        lane_entry = lane_roots.get(root_trans, {})
        
        # Extract English vocabulary meaning
        en_meaning = ""
        if lane_entry:
            # We look for summary_en first, then fall back to English definitions/summaries
            en_meaning = lane_entry.get('summary_en', '')
            if not en_meaning:
                # Clean up general summaries or definitions if summary_en is missing
                en_meaning = f"Root relating to {root_trans}"
        else:
            en_meaning = f"Root relating to {root_trans}"
            
        # Clean and shorten English meaning to represent vocabulary words / core actions
        # Let's extract the first part to make it a concise vocab card
        en_meaning = en_meaning.replace(" (assumed tropical)", "").replace(" (tropical)", "")
        
        # Translate to Indonesian (id_word)
        id_meaning = translate_en_to_id(en_meaning)
        
        # For roots with very long definitions, strip trailing periods and brackets
        en_vocab = en_meaning.strip()
        id_vocab = id_meaning.strip()
        
        insert_records.append((root_ar, en_vocab, id_vocab))
        
        # Print progress and estimate remaining time
        if idx % 100 == 0 or idx == total_roots:
            elapsed = time.time() - start_time
            avg_time = elapsed / idx
            eta = avg_time * (total_roots - idx)
            print(f"Progress: {idx}/{total_roots} roots processed. ETA: {eta:.1f}s")
            
        # Small delay to respect Google Translate web API limits
        time.sleep(0.05)
        
    print("Writing records to learning_harness database...")
    cursor_harness.executemany("""
        INSERT INTO learning_harness (root, en_word, id_word)
        VALUES (?, ?, ?);
    """, insert_records)
    conn_harness.commit()
    
    # Print stats
    cursor_harness.execute("SELECT COUNT(*) FROM learning_harness;")
    loaded_count = cursor_harness.fetchone()[0]
    
    print("\n" + "="*45)
    print("LEARNING HARNESS DATABASE GENERATED!")
    print(f"Database saved to: {HARNESS_DB_PATH}")
    print(f"Total Roots loaded: {loaded_count}")
    print("="*45 + "\n")
    
    # Show first 10 entries as preview
    cursor_harness.execute("SELECT id, root, en_word, id_word FROM learning_harness LIMIT 10;")
    preview = cursor_harness.fetchall()
    print("Preview of first 10 roots in harness:")
    print(f"{'ID':<4} | {'Root':<5} | {'English Meaning':<50} | {'Indonesian Meaning'}")
    print("-"*100)
    for row in preview:
        print(f"{row[0]:<4} | {row[1]:<5} | {row[2][:50]:<50} | {row[3]}")
        
    conn_harness.close()

if __name__ == "__main__":
    main()


--- 
## 🧠 Langkah 9: Kueri Database Pembelajaran Kosakata (Learning Harness)

Sekarang Anda dapat mencari langsung dari database `learning_harness.db` untuk mendapatkan arti kosakata bahasa Inggris dan bahasa Indonesia dari akar kata Arab pilihan Anda.

*Fitur Colab: Masukkan akar kata Arab di kotak input untuk menjalankan kueri (misal: `رحم` atau `قهر`).*

In [ ]:
#@title Kueri Kosakata Dasar (Learning Harness)
root_query = "\u0631\u062d\u0645" #@param {type:"string"}

import sqlite3
conn = sqlite3.connect('learning_harness.db')
cursor = conn.cursor()
cursor.execute("SELECT id, root, en_word, id_word FROM learning_harness WHERE root = ?;", (root_query,))
row = cursor.fetchone()
if row:
    print(f"ID:                {row[0]}")
    print(f"Akar Kata:         {row[1]}")
    print(f"Arti Inggris:      {row[2]}")
    print(f"Arti Indonesia:    {row[3]}")
else:
    print(f"Akar kata '{root_query}' tidak ditemukan di learning_harness.db.")
conn.close()